# psyvis-ml worked example — human vs. models contrast sensitivity (+ a distractor probe)

The PRD §12 lead artifact, end to end: load real `timm` ImageNet classifiers, run the
`ContrastThreshold` suite, fit psychometric functions, and produce the **human-vs-models
contrast figure** (the §5 "killer plot") with the cited human CSF overlay. A shorter second
example runs the **distractor-robustness suite** (a centred target with swept distractors) on
the same models.

## Requirements

- Install the demo extra: `pip install -e ".[demo]"` (adds torch, timm, Pillow).
- Point `IMAGENET_DIR` at **your own** ImageNet subset in ImageFolder layout
  (`<root>/<class>/img.JPEG`). This package does **not** bundle or download ImageNet.
  Class folders may be named by integer ImageNet index (e.g. `207`) or WordNet ID
  (e.g. `n02099601`).

## Read this before interpreting the figure (PRD §14)

The human curve is **grating-detection** contrast sensitivity (Campbell & Robson 1968,
Michelson contrast of a sinusoidal grating). The model curves are **argmax-classification**
correctness on natural images (RMS contrast). These are *different observer paradigms on a
shared contrast axis* — not identical tasks. A model threshold far from the human line is a
difference in what each observer must do, not proof a model does or doesn't "see like" a
human. The figure caption states this; keep it attached to the figure.

In [ ]:
import os

# 1. Demo dependencies (guarded).
try:
    import timm
    import torch
except ImportError as e:
    raise RuntimeError(
        "This notebook needs the demo extra: pip install -e '.[demo]'"
    ) from e

# 2. Your ImageNet subset path (guarded).
IMAGENET_DIR = os.environ.get("IMAGENET_DIR", "/path/to/your/imagenet_subset")
if not os.path.isdir(IMAGENET_DIR):
    raise RuntimeError(
        f"Set IMAGENET_DIR to your own ImageFolder-style ImageNet subset; got "
        f"{IMAGENET_DIR!r}. This package does not bundle or download ImageNet — see the\n"
        "cell above for the expected layout."
    )

import matplotlib.pyplot as plt

import psyvis_ml as pe

print("psyvis-ml", pe.__version__, "| torch", torch.__version__, "| timm", timm.__version__)

In [ ]:
# Load a small labeled subset and build 2-3 real classifiers as plain callables.
ds = pe.datasets.imagenet_subset(IMAGENET_DIR, max_classes=5, max_per_class=40,
                                 image_size=224)
print("loaded", len(ds.images), "images across", len(ds.metadata["classes"]), "classes")

MODEL_NAMES = ["resnet18", "resnet50", "mobilenetv3_small_100"]
models = {name: pe.models.timm_classifier(name, pretrained=True) for name in MODEL_NAMES}

In [ ]:
# Sweep contrast and fit a psychometric function per model (ground-truth accuracy).
levels = pe.linspace_levels(0.01, 0.5, 9, spacing="log")   # target RMS contrast
suite = pe.suites.ContrastThreshold(contrast_metric="rms", clip_range=(0.0, 1.0))

results = [
    pe.measure(model=clf, suite=suite, dataset=ds, levels=levels, seed=0, model_name=name)
    for name, clf in models.items()
]
for r in results:
    print(f"{r.model_name}: 75% contrast threshold = {r.threshold(0.75):.4g}")

In [ ]:
# The killer plot: several models + the cited human CSF on one contrast axis.
fig = results[0].compare(results[1:], human="auto", n_boot=300, seed=0)
fig.set_size_inches(7.5, 5.2)
plt.show()

The models produce **curves, not points**: read off each model's contrast threshold and slope
and how far it sits from the human reference. Remember the paradigm caveat in the caption —
the large human-vs-model offset mixes a real sensitivity difference with the
detection-vs-classification / Michelson-vs-RMS mismatch.

In [ ]:
# Second example: the distractor-robustness suite. A CENTRED target (where the classifier's
# isolated-object ceiling holds) is surrounded by distractor copies at a swept spacing;
# P(correct) vs. spacing measures how a nearby distractor's distance degrades recognition.
# This is a robustness measure, not a model of human crowding.
ds_small = pe.datasets.imagenet_subset(IMAGENET_DIR, max_classes=5, max_per_class=40,
                                       image_size=96)
distractor_suite = pe.suites.DistractorRobustness(canvas_shape=(224, 224),
                                                  include_baseline=True)
spacings = pe.linspace_levels(40, 110, 7)
dr = pe.measure(model=models["resnet50"], suite=distractor_suite, dataset=ds_small,
                levels=spacings, seed=0, model_name="resnet50")
fig2 = dr.plot(n_boot=150, seed=0)
fig2.set_size_inches(7.0, 4.6)
plt.show()
print("distractor-distance threshold (px):",
      {lbl: round(float(v), 2) for lbl, v in dr.threshold(0.75).items()})

In [ ]:
# A self-contained methods bundle (table + figures + reproducibility metadata).
bundle = results[0].report(others=results[1:], outdir="report_bundle", human="auto")
print("wrote report to:", bundle.directory)
for p in bundle.artifacts():
    print("  ", p.relative_to(bundle.directory))

## Notes / honest caveats

- A handful of classes and a few hundred images gives **coarse** curves and wide CIs; scale up
  your subset for publication-grade thresholds.
- Contrast is set in **RMS** here; the human reference is in **Michelson** grating contrast.
  Treat the human-vs-model offset qualitatively, not as a calibrated ratio.
- The distractor-robustness suite is a **robustness** measure (how a nearby distractor's
  distance degrades a *centred* classifier), **not** a model of human crowding: the target is
  central and the threshold is a distractor-distance tolerance, not a perceptual critical
  spacing. Keep the target large/central enough that its isolated-object accuracy is high,
  or the curve measures target loss rather than distractor interference.